# 85) Colab Adapter Training (T4)

This notebook trains a LoRA adapter on free Colab GPU, then packages the adapter for download.

After download, place `lora_adapter` into `artifacts/runs/finetune_unsloth/lora_adapter` in your local repo.

In [6]:
# 1) Confirm GPU runtime (Runtime -> Change runtime type -> T4 GPU)
!nvidia-smi


Tue Mar 31 19:27:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             13W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [7]:
# 2) Clone repo + move into it
from pathlib import Path

repo_dir = Path('/content/polyglot-grounded-qa')
if not repo_dir.exists():
    !git clone https://github.com/aaliyan1230/polyglot-grounded-qa.git /content/polyglot-grounded-qa

%cd /content/polyglot-grounded-qa
!git pull --ff-only

/content/polyglot-grounded-qa
Already up to date.


In [8]:
# 3) Install dependencies
!pip -q install -U uv
!pip -q install unsloth datasets trl peft accelerate bitsandbytes
!uv sync --extra finetune

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 82.3 MB/s eta 0:00:00:00:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.5/55.5 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.9/62.9 MB 10.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 49.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 19.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 22.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.7/10.7 MB 115.2 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 403.8/403.8 kB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 102.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 109.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.7/183.7 kB 23.1 MB/s eta

In [ ]:
# 4) Build finetune dataset + splits + formatted training files
!uv run python scripts/build_sft_dataset.py
!uv run python scripts/split_sft_dataset.py
!uv run python scripts/format_sft_for_training.py

!ls -lah data/benchmarks/finetune
!ls -lah data/benchmarks/finetune/formatted

Traceback (most recent call last):
  File "/content/polyglot-grounded-qa/scripts/format_sft_for_training.py", line 120, in <module>
    main()
  File "/content/polyglot-grounded-qa/scripts/format_sft_for_training.py", line 109, in main
    rows = _read_jsonl(in_dir / f"{split}.jsonl")
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/polyglot-grounded-qa/scripts/format_sft_for_training.py", line 11, in _read_jsonl
    with path.open("r", encoding="utf-8") as f:
         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/pathlib.py", line 1013, in open
    return io.open(self, mode, buffering, encoding, errors, newline)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
FileNotFoundError: [Errno 2] No such file or directory: '/content/polyglot-grounded-qa/data/benchmarks/finetune/train.jsonl'
ls: cannot access 'data/benchmarks/finetune/formatted': No such file or directory


In [ ]:
# Optional fallback if split files are still missing
from pathlib import Path

required = [
    Path('data/benchmarks/finetune/train.jsonl'),
    Path('data/benchmarks/finetune/val.jsonl'),
    Path('data/benchmarks/finetune/test.jsonl'),
]

missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Missing split files, running end-to-end data pipeline:', missing)
    !uv run python scripts/run_finetune_data_pipeline.py
else:
    print('All split files present.')

In [ ]:
# 5) Create a Colab override config (shorter run for free sessions)
from pathlib import Path
import yaml

base_cfg = Path('configs/finetune/cloud_unsloth_qlora.yaml')
colab_cfg = Path('configs/finetune/colab_unsloth_qlora.yaml')

cfg = yaml.safe_load(base_cfg.read_text(encoding='utf-8'))
cfg['training']['max_steps'] = 200
cfg['training']['save_steps'] = 50
cfg['training']['logging_steps'] = 10
cfg['run_name'] = 'colab-free-unsloth-qlora'

colab_cfg.write_text(yaml.safe_dump(cfg, sort_keys=False), encoding='utf-8')
print('Wrote', colab_cfg)

In [ ]:
# 6) Train adapter
!uv run python scripts/train_unsloth_sft.py \
  --config configs/finetune/colab_unsloth_qlora.yaml \
  --train-file data/benchmarks/finetune/formatted/train.text.jsonl \
  --val-file data/benchmarks/finetune/formatted/val.text.jsonl \
  --output-dir artifacts/runs/finetune_unsloth

In [ ]:
# 7) Verify + package adapter for download
from pathlib import Path

adapter_dir = Path('artifacts/runs/finetune_unsloth/lora_adapter')
zip_path = Path('/content/lora_adapter.zip')

if not adapter_dir.exists():
    raise FileNotFoundError(f'Missing adapter directory: {adapter_dir}')

!cd artifacts/runs/finetune_unsloth && zip -r /content/lora_adapter.zip lora_adapter
print('Packaged:', zip_path)
print('Download this zip from Colab Files pane.')

## Local follow-up (back in VS Code)

1. Unzip into `artifacts/runs/finetune_unsloth/lora_adapter`.
2. Run:

```bash
uv run python scripts/run_trained_adapter_eval.py \
  --variant tuned-adapter-v1 \
  --base-model Qwen/Qwen2.5-3B-Instruct \
  --adapter-path artifacts/runs/finetune_unsloth/lora_adapter \
  --append
```

3. Re-run final notebook comparison/takeaway cells in `notebooks/80_final_results.ipynb`.